# Modelo de Previsão de Tickets Totais

### Objetivo

Desenvolver um modelo de previsão para a quantidade total de tickets, utilizando dados históricos e métricas relevantes.
O modelo deve ser capaz de prever a demanda futura de tickets mensal, auxiliando na alocação de recursos e planejamento operacional.

### Passos do Projeto

1. **Análise Exploratória de Dados (EDA)**: Compreender a distribuição dos dados, identificar padrões sazonais e tendências.
2. **Pré-processamento de Dados**: Limpeza dos dados, tratamento de valores ausentes e transformação de variáveis.
3. **Seleção de Modelos**: Avaliar diferentes algoritmos de previsão, como ARIMA, Prophet, LSTM, entre outros.
4. **Treinamento e Validação**: Treinar os modelos selecionados e validar seu desempenho utilizando métricas como MAE, RMSE

In [13]:
import sys
import os
import warnings

import pandas as pd
import numpy as np
import mlflow

# imports modelos
from prophet import Prophet
from lightgbm import LGBMRegressor
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
sys.path.insert(0, project_root)

In [12]:
from src.utils.extract_data import get_data

In [2]:
mlflow.set_tracking_uri("https://visiondata.ininetech.com.br/mlflow")
mlflow.set_experiment("run_tickets_totais")
mlflow.autolog(disable=True)

## Extração de dados

In [3]:
df = get_data("../../sql/all_dates2.sql")

## Tratamento de dados

In [4]:
# Converter para datetime e definir como índice
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date")
df = df.sort_index()

In [5]:
print(f"Shape dos dados: {df.shape}")
print(f"Período: {df.index.min()} até {df.index.max()}")
print("Informações: ", df.info())
print("\nEstatísticas:")
print(df.describe())

Shape dos dados: (2064, 1)
Período: 2020-01-06 00:00:00 até 2025-08-30 00:00:00
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2064 entries, 2020-01-06 to 2025-08-30
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   QtTickets  2064 non-null   int64
dtypes: int64(1)
memory usage: 32.2 KB
Informações:  None

Estatísticas:
         QtTickets
count  2064.000000
mean     43.953973
std       6.719381
min      21.000000
25%      39.000000
50%      44.000000
75%      49.000000
max      67.000000


## Separação entre treino e teste

In [6]:
train_size = int(len(df) * 0.8)
train, test = df.iloc[:train_size], df.iloc[train_size:]

In [7]:
df = df.asfreq("D")  # força frequência diária
df["QtTickets"] = df["QtTickets"].interpolate()  # preencher valores ausentes

## Funções de avaliação

In [8]:
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, rmse, r2, mape

In [9]:
# AR
def run_ar(train, test):
    model = ARIMA(train, order=(1, 0, 0)).fit()
    preds = model.forecast(steps=len(test))
    return model, preds


# MA
def run_ma(train, test):
    model = ARIMA(train, order=(0, 0, 1)).fit()
    preds = model.forecast(steps=len(test))
    return model, preds


# ARMA
def run_arma(train, test):
    model = ARIMA(train, order=(2, 0, 2)).fit()
    preds = model.forecast(steps=len(test))
    return model, preds


# ARIMA
def run_arima(train, test):
    model = ARIMA(train, order=(2, 1, 2)).fit()
    preds = model.forecast(steps=len(test))
    return model, preds


# SARIMA
def run_sarima(train, test):
    model = SARIMAX(train, order=(2, 1, 2), seasonal_order=(1, 1, 1, 12)).fit()
    preds = model.forecast(steps=len(test))
    return model, preds


# SARIMAX
def run_sarimax(train, test, exog_train=None, exog_test=None):
    model = SARIMAX(
        train, order=(2, 1, 2), seasonal_order=(1, 1, 1, 12), exog=exog_train
    ).fit()
    preds = model.forecast(steps=len(test), exog=exog_test)
    return model, preds


# Holt-Winters
def run_holt_winters(train, test):
    model = ExponentialSmoothing(
        train, trend="add", seasonal="add", seasonal_periods=12
    ).fit()
    preds = model.forecast(steps=len(test))
    return model, preds


# Prophet
def run_prophet(train, test):
    prophet_df = train.reset_index().rename(columns={"Date": "ds", "QtTickets": "y"})
    model = Prophet()
    model.fit(prophet_df)
    future = model.make_future_dataframe(periods=len(test))
    forecast = model.predict(future)
    preds = forecast.set_index("ds")["yhat"][-len(test) :]
    return model, preds


# LightGBM
def run_lightgbm(train, test):
    lgbm = LGBMRegressor(n_estimators=1000)
    lgbm.fit(np.arange(len(train)).reshape(-1, 1), train.values)
    preds = lgbm.predict(np.arange(len(train), len(train) + len(test)).reshape(-1, 1))
    return lgbm, preds

In [10]:
# dicionario de modelos
model = {
    "AR": run_ar,
    "MA": run_ma,
    "ARMA": run_arma,
    "ARIMA": run_arima,
    "SARIMA": run_sarima,
    "SARIMAX": run_sarimax,
    "Holt-Winters": run_holt_winters,
    "Prophet": run_prophet,
    "LightGBM": run_lightgbm,
}

In [ ]:
# treinar e avaliar cada modelo
list_metrics = {}  # dicionário para armazenar as métricas de cada modelo

for name, func in model.items():
    with mlflow.start_run(run_name=name):
        # treinamento e previsão
        model_fit, preds = func(train["QtTickets"], test["QtTickets"])

        # metricas avaliação
        mae, rmse, r2, mape = evaluate_model(test["QtTickets"], preds)

        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)
        mlflow.log_metric("MAPE", mape)

        # salvar previsoes como artefato .csv e .pkl
        result_df = pd.DataFrame({"Real": test["QtTickets"], "Previsto": preds})
        csv_path = f"../../../data/{name}_predicoes.csv"
        result_df.to_csv(csv_path, index=False)
        model_fit_path = f"../../../models/{name}_model.pkl"
        pd.to_pickle(model_fit, model_fit_path)

🏃 View run AR at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1/runs/5a2ac547adee4f299de37c2ea72cde5b
🧪 View experiment at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1
🏃 View run MA at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1/runs/d940d43d163f41a19829e959aa78f2df
🧪 View experiment at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1
🏃 View run ARMA at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1/runs/10e07a0ef0244d8783cdc4be3bcbf15a
🧪 View experiment at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1
🏃 View run ARIMA at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1/runs/2c41da1052034240aae918c8c1baaf68
🧪 View experiment at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1
🏃 View run SARIMA at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1/runs/0688597c947d48f0a9e348a210642a05
🧪 View experiment at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1
🏃 View run 

21:59:52 - cmdstanpy - INFO - Chain [1] start processing


🏃 View run Holt-Winters at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1/runs/1f3377b6d2bb41b7b6df7396d8c89403
🧪 View experiment at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1


21:59:52 - cmdstanpy - INFO - Chain [1] done processing


🏃 View run Prophet at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1/runs/6d4b9501a7d3454fbf1e263c78da6770
🧪 View experiment at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000126 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 1651, number of used features: 1
[LightGBM] [Info] Start training from score 43.901878
🏃 View run LightGBM at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1/runs/b96ac11fa9e44c2f8e70fc6ae3af6e7d
🧪 View experiment at: https://visiondata.ininetech.com.br/mlflow/#/experiments/1
